In [1]:
import os
import time
import random
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, balanced_accuracy_score

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)

In [2]:
train_df = pd.read_csv('train_data_bulk_trial.csv')
test_df = pd.read_csv('test_data_bulk_trial.csv')

In [3]:
TEXT_COL = "text"
LABEL_COLS = ["target_1", "target_2", "target_3", "target_4"]
DIM_NAMES = LABEL_COLS

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

In [4]:
train_df = train_df.copy()
test_df  = test_df.copy()

train_df[TEXT_COL] = train_df[TEXT_COL].astype(str)
test_df[TEXT_COL]  = test_df[TEXT_COL].astype(str)

y_train_full = train_df[LABEL_COLS].astype(int).values
y_test = test_df[LABEL_COLS].astype(int).values

tr_df, va_df = train_test_split(train_df, test_size=0.2, random_state=SEED)

In [5]:
def compute_pos_weight(df: pd.DataFrame, label_cols):
    # pos_weight = N_neg / N_pos for each label dimension
    y = df[label_cols].astype(int).values
    pos = y.sum(axis=0)
    neg = y.shape[0] - pos
    # avoid division by zero
    pos = np.clip(pos, 1, None)
    pos_weight = (neg / pos).astype(np.float32)
    return torch.tensor(pos_weight)

pos_weight = compute_pos_weight(tr_df, LABEL_COLS)
pos_weight

tensor([2.5212, 1.8988, 1.3540, 1.7675])

In [6]:
def to_hf_dataset(df: pd.DataFrame) -> Dataset:
    return Dataset.from_pandas(df[[TEXT_COL] + LABEL_COLS].reset_index(drop=True))

ds_train = to_hf_dataset(tr_df)
ds_val   = to_hf_dataset(va_df)
ds_test  = to_hf_dataset(test_df)

from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding

MODEL_NAME_C = "microsoft/deberta-v3-base"

tokenizer_c = AutoTokenizer.from_pretrained(MODEL_NAME_C)
data_collator_c = DataCollatorWithPadding(tokenizer=tokenizer_c)

MAX_LEN_C = 512   # since you want full posts

/home/nina/venv/lib/python3.11/site-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [7]:
def tokenize_batch_c(batch):
    return tokenizer_c(
        batch[TEXT_COL],
        truncation=True,
        max_length=MAX_LEN_C,
    )
ds_train_tok_c = ds_train.map(tokenize_batch_c, batched=True)
ds_val_tok_c   = ds_val.map(tokenize_batch_c, batched=True)
ds_test_tok_c  = ds_test.map(tokenize_batch_c, batched=True)

# Trainer expects "labels" field; for multi-label make it float tensor
def add_labels(batch):
    labels = np.stack([batch[c] for c in LABEL_COLS], axis=1).astype(np.float32)
    batch["labels"] = labels
    return batch

ds_train_tok_c = ds_train_tok_c.map(add_labels, batched=True)
ds_val_tok_c   = ds_val_tok_c.map(add_labels, batched=True)
ds_test_tok_c  = ds_test_tok_c.map(add_labels, batched=True)

cols_to_remove = [TEXT_COL] + LABEL_COLS

ds_train_tok_c = ds_train_tok_c.remove_columns(cols_to_remove)
ds_val_tok_c   = ds_val_tok_c.remove_columns(cols_to_remove)
ds_test_tok_c  = ds_test_tok_c.remove_columns(cols_to_remove)


Map:   0%|          | 0/71850 [00:00<?, ? examples/s]

Map:   0%|          | 0/17963 [00:00<?, ? examples/s]

Map:   0%|          | 0/22454 [00:00<?, ? examples/s]

Map:   0%|          | 0/71850 [00:00<?, ? examples/s]

Map:   0%|          | 0/17963 [00:00<?, ? examples/s]

Map:   0%|          | 0/22454 [00:00<?, ? examples/s]

In [8]:


data_collator = DataCollatorWithPadding(tokenizer=tokenizer_c)

In [9]:
def fulltype_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    correct = (y_true == y_pred).sum(axis=1)
    return {
        "full_type_acc_all4": float((correct == 4).mean()),
        "at_least_3": float((correct >= 3).mean()),
        "at_least_2": float((correct >= 2).mean()),
        "at_least_1": float((correct >= 1).mean()),
    }

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)
    labels = labels.astype(int)

    # per-dim metrics
    macro_f1s = []
    bal_accs = []
    for k in range(labels.shape[1]):
        macro_f1s.append(f1_score(labels[:, k], preds[:, k], average="macro"))
        bal_accs.append(balanced_accuracy_score(labels[:, k], preds[:, k]))

    out = {
        "macro_f1_mean": float(np.mean(macro_f1s)),
        "balanced_acc_mean": float(np.mean(bal_accs)),
    }
    out.update(fulltype_metrics(labels, preds))
    return out

In [10]:
class WeightedBCETrainer(Trainer):
    def __init__(self, pos_weight: torch.Tensor, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weight = pos_weight

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels").to(model.device)  # float32 shape (B,4)
        outputs = model(**inputs)
        logits = outputs.logits

        loss_fct = torch.nn.BCEWithLogitsLoss(pos_weight=self.pos_weight.to(model.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

In [11]:
# MODEL_NAME = "roberta-base"
# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME_C)

model_c = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME_C,
    num_labels=4,
    problem_type="multi_label_classification",
).to(DEVICE)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer_c)

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:

args_c = TrainingArguments(
    output_dir="outputs_deberta",

    eval_strategy="steps",
    eval_steps=1000,

    save_strategy="steps",
    save_steps=1000,

    logging_steps=100,

    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1_mean",
    greater_is_better=True,

    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,

    num_train_epochs=3,
    fp16=True,

    seed=SEED,
    report_to="none",
)

trainer_c = WeightedBCETrainer(
    pos_weight=pos_weight,
    model=model_c,
    args=args_c,
    train_dataset=ds_train_tok_c,
    eval_dataset=ds_val_tok_c,
    tokenizer=tokenizer_c,
    data_collator=data_collator_c,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=4)],
)
trainer_c.train()

/tmp/ipykernel_3847000/743699552.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeightedBCETrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)


Step,Training Loss,Validation Loss,Macro F1 Mean,Balanced Acc Mean,Full Type Acc All4,At Least 3,At Least 2,At Least 1
1000,0.903400,0.893976,0.433614,0.544912,0.089016,0.378612,0.761844,0.959751
2000,0.746800,0.699158,0.741849,0.734179,0.436731,0.749930,0.924511,0.991817
3000,0.621500,0.618333,0.779637,0.779670,0.544397,0.760619,0.919835,0.989701
4000,0.559000,0.595854,0.788872,0.799136,0.560374,0.755776,0.911540,0.986083
5000,0.502200,0.620910,0.798007,0.803003,0.583867,0.775873,0.917720,0.987196
6000,0.566700,0.688909,0.802965,0.804247,0.590659,0.780048,0.926850,0.989256
7000,0.576600,0.650148,0.776430,0.794383,0.545232,0.734955,0.898291,0.981128
8000,0.503400,0.558895,0.816167,0.818893,0.613762,0.797473,0.931192,0.990146
9000,0.515900,0.504795,0.821064,0.812779,0.612704,0.818961,0.947336,0.993821
10000,0.593500,0.493656,0.812003,0.824796,0.598619,0.785671,0.924734,0.987586


In [19]:
test_metrics = trainer_c.evaluate(ds_test_tok_c)
test_metrics

{'eval_loss': 0.4842887222766876,
 'eval_macro_f1_mean': 0.8239429743178206,
 'eval_balanced_acc_mean': 0.828863756126798,
 'eval_full_type_acc_all4': 0.6197114099937651,
 'eval_at_least_3': 0.8054689587601318,
 'eval_at_least_2': 0.9385855526854904,
 'eval_at_least_1': 0.9914491850004453}

In [20]:
pred_output = trainer_c.predict(ds_test_tok_c)

logits = pred_output.predictions
labels = pred_output.label_ids

In [21]:
labels

array([[0., 0., 1., 1.],
       [0., 0., 0., 0.],
       [0., 0., 0., 1.],
       ...,
       [0., 0., 0., 1.],
       [1., 0., 1., 0.],
       [0., 1., 0., 0.]], dtype=float32)

In [23]:
probs = 1 / (1 + np.exp(-logits))
preds = (probs >= 0.5).astype(int)

In [26]:
labels = labels.astype(int)

In [27]:
def bits_to_int(y_bits):
    return (
        (y_bits[:,0] << 3) +
        (y_bits[:,1] << 2) +
        (y_bits[:,2] << 1) +
        (y_bits[:,3] << 0)
    )

y_true_type = bits_to_int(labels)
y_pred_type = bits_to_int(preds)

In [28]:
from sklearn.metrics import f1_score, accuracy_score

macro_f1_16 = f1_score(y_true_type, y_pred_type, average="macro")
acc_16 = accuracy_score(y_true_type, y_pred_type)

print("16-type Accuracy:", acc_16)
print("16-type Macro F1:", macro_f1_16)

16-type Accuracy: 0.6197114099937651
16-type Macro F1: 0.5929435798574987


In [29]:
save_dir = "deberta_mbti_finetuned_2_earlys4"

trainer_c.save_model(save_dir)
tokenizer_c.save_pretrained(save_dir)

('deberta_mbti_finetuned_2_earlys4/tokenizer_config.json',
 'deberta_mbti_finetuned_2_earlys4/special_tokens_map.json',
 'deberta_mbti_finetuned_2_earlys4/spm.model',
 'deberta_mbti_finetuned_2_earlys4/added_tokens.json',
 'deberta_mbti_finetuned_2_earlys4/tokenizer.json')